In [1]:
# ==========================================
# Author: George Ayensu, PE
# Project: PhD Working Code - Week 3 (Corrected)
# Case Study: Data Center Reservoir Thermal Energy Storage (RTES)
# Date: November 2026
# ==========================================

!pip install pyomo highspy -q
import pyomo.environ as pyo

model = pyo.ConcreteModel()
model.T = pyo.Set(initialize=range(1, 25))

# --- DATA CENTER RTES PARAMETERS ---
def load_rule(model, t):
    if t >= 22 or t <= 6: return 8000.0  # Night drop
    else: return 15000.0                 # Day peak
model.cooling_load = pyo.Param(model.T, initialize=load_rule)

max_chiller_power = 4000.0
max_rtes_capacity = 60000.0

traditional_cop = 4.0
rtes_cop = 16.5  # The RTES heat pump is super efficient!

def price_rule(model, t):
    if t <= 6 or t >= 22: return 0.10
    else: return 0.30
model.price = pyo.Param(model.T, initialize=price_rule)

# Variables
model.chiller_elec = pyo.Var(model.T, within=pyo.NonNegativeReals)
model.rtes_elec = pyo.Var(model.T, within=pyo.NonNegativeReals) # NEW: Electricity to charge RTES
model.rtes_charge = pyo.Var(model.T, within=pyo.NonNegativeReals)
model.rtes_discharge = pyo.Var(model.T, within=pyo.NonNegativeReals)
model.rtes_soc = pyo.Var(model.T, within=pyo.NonNegativeReals)

# Constraints
model.limit_chiller = pyo.Constraint(model.T, rule=lambda m, t: m.chiller_elec[t] <= max_chiller_power)
model.limit_rtes = pyo.Constraint(model.T, rule=lambda m, t: m.rtes_soc[t] <= max_rtes_capacity)

# 1. Cooling Load met by Chiller + RTES Discharge
def cooling_balance_rule(model, t):
    return (model.chiller_elec[t] * traditional_cop) + model.rtes_discharge[t] == model.cooling_load[t]
model.cooling_balance = pyo.Constraint(model.T, rule=cooling_balance_rule)

# 2. NEW: RTES Charge is created by RTES Electricity (using high COP)
def rtes_charge_rule(model, t):
    return model.rtes_charge[t] == model.rtes_elec[t] * rtes_cop
model.rtes_charge_constraint = pyo.Constraint(model.T, rule=rtes_charge_rule)

# 3. RTES Storage Balance
def rtes_balance_rule(model, t):
    if t == 1:
        return model.rtes_soc[t] == 0 + model.rtes_charge[t] - model.rtes_discharge[t]
    else:
        return model.rtes_soc[t] == model.rtes_soc[t-1] + model.rtes_charge[t] - model.rtes_discharge[t]
model.rtes_balance = pyo.Constraint(model.T, rule=rtes_balance_rule)

# Objective: Minimize Electricity Cost (Chiller + RTES Heat Pump)
model.cost = pyo.Objective(
    expr=sum(model.price[t] * (model.chiller_elec[t] + model.rtes_elec[t]) for t in model.T),
    sense=pyo.minimize
)

# Solve
solver = pyo.SolverFactory('appsi_highs')
solver.solve(model)

print(f"Total Daily Electricity Cost (Corrected RTES): ${model.cost.expr():,.2f}")
print("-------------------------------------------------------------------------")
print("Hour | Price | Cooling Load | Chiller Elec | RTES Elec | RTES Discharge | RTES SOC")
for t in model.T:
    print(f"{t:4} | ${model.price[t]:4.2f} | {model.cooling_load[t]:12.1f} | {model.chiller_elec[t].value:12.1f} | {model.rtes_elec[t].value:9.1f} | {model.rtes_discharge[t].value:14.1f} | {model.rtes_soc[t].value:8.1f}")


Total Daily Electricity Cost (Corrected RTES): $3,800.00
-------------------------------------------------------------------------
Hour | Price | Cooling Load | Chiller Elec | RTES Elec | RTES Discharge | RTES SOC
   1 | $0.10 |       8000.0 |          0.0 |     484.8 |         8000.0 |      0.0
   2 | $0.10 |       8000.0 |          0.0 |     484.8 |         8000.0 |      0.0
   3 | $0.10 |       8000.0 |          0.0 |     484.8 |         8000.0 |      0.0
   4 | $0.10 |       8000.0 |          0.0 |     484.8 |         8000.0 |      0.0
   5 | $0.10 |       8000.0 |          0.0 |     484.8 |         8000.0 |      0.0
   6 | $0.10 |       8000.0 |          0.0 |    4121.2 |         8000.0 |  60000.0
   7 | $0.30 |      15000.0 |          0.0 |      -0.0 |        15000.0 |  45000.0
   8 | $0.30 |      15000.0 |          0.0 |      -0.0 |        15000.0 |  30000.0
   9 | $0.30 |      15000.0 |          0.0 |      -0.0 |        15000.0 |  15000.0
  10 | $0.30 |      15000.0 |          